<a href="https://colab.research.google.com/github/24dhanush/GUVI_PROJECT/blob/project-4/bitcoinprediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import math
import datetime as dt

# For Evalution we will use these library

from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from sklearn.preprocessing import MinMaxScaler

# For PLotting we will use these library

import matplotlib.pyplot as plt
from itertools import cycle
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [ ]:
maindf=pd.read_csv('/content/Bitcoin Historical Data (1).csv')

In [ ]:
maindf.head()

In [ ]:
maindf['Date'] = pd.to_datetime(maindf['Date'], format='%d-%m-%Y')

y_overall = maindf.loc[(maindf['Date'] >= '18-07-2010')
                     & (maindf['Date'] <= '24-03-2024')]

y_overall = y_overall.drop(columns=['Change %','Vol.'])

In [ ]:
y_overall['Price'] = y_overall['Price'].str.replace(',', '').astype(float)
y_overall['Open'] = y_overall['Open'].str.replace(',', '').astype(float)

monthvise= y_overall.groupby(y_overall['Date'].dt.strftime('%B'))[['Price','Open']].mean()
new_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August',
             'September', 'October', 'November', 'December']
monthvise = monthvise.reindex(new_order, axis=0)
monthvise

In [ ]:
names = cycle(['Stock Open Price','Stock Close Price','Stock High Price','Stock Low Price'])

y_overall['High'] = y_overall['High'].str.replace(',', '').astype(float)
y_overall['Low'] = y_overall['Low'].str.replace(',', '').astype(float)

fig = px.line(y_overall, x=y_overall.Date, y=[y_overall['Open'], y_overall['Price'],
                                          y_overall['High'], y_overall['Low']],
             labels={'Date': 'Date','value':'Stock value'})
fig.update_layout(title_text='Stock analysis chart', font_size=15, font_color='black',legend_title_text='Stock Parameters')
fig.for_each_trace(lambda t:  t.update(name = next(names)))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

In [ ]:
#initalisation and split past 60 data
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models

def create_sequences(data, window_size=60, horizon=1):
    X, y = [], []
    for i in range(window_size, len(data) - horizon + 1):
        X.append(data[i-window_size:i])
        y.append(data[i:i+horizon])
    return np.array(X), np.array(y)

In [ ]:
df = pd.read_csv('/content/Bitcoin Historical Data (1).csv')
df['Price'] = df['Price'].str.replace(',', '').astype(float)
data = df[['Price']].values
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

In [ ]:
X, y = create_sequences(scaled_data, window_size=60, horizon=3)

MODELS LIST

In [ ]:
#LSTM
def build_lstm(input_shape, horizon):
    model = models.Sequential([
        layers.LSTM(64, return_sequences=False, input_shape=input_shape),
        layers.Dense(32, activation='relu'),
        layers.Dense(horizon)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
#RNN
def build_rnn(input_shape, horizon):
    model = models.Sequential([
        layers.SimpleRNN(50, input_shape=input_shape),
        layers.Dense(horizon)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
#1D CNN
def build_cnn(input_shape, horizon):
    model = models.Sequential([
        layers.Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(50, activation='relu'),
        layers.Dense(horizon)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
#Transforme
def build_transformer(input_shape, horizon):
    inputs = layers.Input(shape=input_shape)

    # Self-Attention Layer
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=input_shape[-1])(inputs, inputs)
    x = layers.LayerNormalization()(attention + inputs)

    # Feed Forward
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(horizon)(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
# Split data (e.g., 80% train, 20% test)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

lstm_model = build_lstm(input_shape=(60, 1), horizon=3)
rnn_model = build_rnn(input_shape=(60, 1), horizon=3)
cnn_model = build_cnn(input_shape=(60, 1), horizon=3)
transformer_model = build_transformer(input_shape=(60, 1), horizon=3)
history_lstm = lstm_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))
history_rnn = rnn_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))
history_cnn = cnn_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))
history_transformer = transformer_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

In [ ]:
predictions_scaled_lstm = lstm_model.predict(X_test)
predictions_scaled_rnn = rnn_model.predict(X_test)
predictions_scaled_cnn = cnn_model.predict(X_test)
predictions_scaled_transformer = transformer_model.predict(X_test)


In [ ]:
predictions_actual_lstm = scaler.inverse_transform(predictions_scaled_lstm.reshape(-1, 1))
predictions_actual_rnn = scaler.inverse_transform(predictions_scaled_rnn.reshape(-1, 1))
predictions_actual_cnn = scaler.inverse_transform(predictions_scaled_cnn.reshape(-1, 1))
predictions_actual_transformer = scaler.inverse_transform(predictions_scaled_transformer.reshape(-1, 1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

**EVALUATION**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

def evaluate_model_performance(y_actual, y_predicted, model_name, horizon):
    # Calculate Metrics as required by project [1]
    mae = mean_absolute_error(y_actual, y_predicted)
    rmse = np.sqrt(mean_squared_error(y_actual, y_predicted))
    mape = mean_absolute_percentage_error(y_actual, y_predicted) * 100

    print(f"--- Evaluation for {model_name} ({horizon}-Day Horizon) ---")
    print(f"Mean Absolute Error (MAE): ${mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}


In [ ]:
evaluate_model_performance(y_test_actual,predictions_actual_lstm,"LSTM_MODEL",horizon=3)

In [ ]:
evaluate_model_performance(y_test_actual,predictions_actual_rnn,"RNN_MODEL",horizon=3)

In [ ]:
evaluate_model_performance(y_test_actual,predictions_actual_cnn,"CNN_MODEL",horizon=3)

In [ ]:
evaluate_model_performance(y_test_actual,predictions_actual_transformer,"TRANSFORMER_MODEL",horizon=3)

VISUALITION

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_forecast_results(y_actual, y_predicted, history, model_name):
    plt.figure(figsize=(15, 10))

    # 1. Actual vs Predicted Price Curve [1]
    plt.subplot(2, 1, 1)
    plt.plot(y_actual, label="Actual Price", color='blue', linewidth=2)
    plt.plot(y_predicted, label="Predicted Price", color='orange', linestyle='--', linewidth=2)
    plt.title(f"{model_name}: Actual vs Predicted Bitcoin Price")
    plt.xlabel("Time (Days)")
    plt.ylabel("Price (USD)")
    plt.legend()

    # 2. Training & Validation Loss Curve [2]
    plt.subplot(2, 1, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f"{model_name} Learning Curve (Loss)")
    plt.xlabel("Epochs")
    plt.ylabel("Loss (MSE)")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_forecast_results(y_test_actual,predictions_actual_lstm,history_lstm,'LSTM Model')

In [ ]:
plot_forecast_results(y_test_actual,predictions_actual_rnn,history_rnn,'RNN Model')

In [ ]:
plot_forecast_results(y_test_actual,predictions_actual_cnn,history_cnn,'CNN Model')

In [ ]:
plot_forecast_results(y_test_actual,predictions_actual_transformer,history_transformer,'Transformer Model')

PREDICTION

In [ ]:
import numpy as np

def predict_future_price(model, recent_data, scaler, horizon):
    if len(recent_data) != 60:
        raise ValueError("The model requires exactly 60 days of past data.")
    input_array = np.array(recent_data).reshape(-1, 1)
    scaled_input = scaler.transform(input_array)
    final_input = scaled_input.reshape((1, 60, 1))
    scaled_prediction = model.predict(final_input)
    actual_prediction = scaler.inverse_transform(scaled_prediction)
    return actual_prediction.flatten()



In [ ]:
# Get the last 60 data points from the original scaled data for prediction input
last_60_days = scaled_data[-60:].flatten().tolist()

# Use the 'model' variable for the trained LSTM model and 'scaler' for the MinMaxScaler
forecast = predict_future_price(lstm_model, last_60_days, scaler, horizon=3)

print(f"The predicted prices for the next 3 days are: {forecast}")

In [ ]:
pip install streamlit

In [ ]:
%%writefile bitcoinapp.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow.keras import layers, models

# --------------------------------------------------
# PAGE CONFIG
# --------------------------------------------------
st.set_page_config(
    page_title="Bitcoin Prediction",
    layout="wide"
)

# --------------------------------------------------
# TITLE
# --------------------------------------------------
st.title("📈 Project 4 - Bitcoin Price Prediction")

st.markdown("---")

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------
st.sidebar.header("Model Selection")

selected_model = st.sidebar.selectbox(
    "Choose Prediction Model",
    [
        "LSTM",
        "RNN",
        "CNN",
        "Transformer"
    ]
)

epochs = st.sidebar.slider(
    "Epochs",
    5,
    50,
    10
)

batch_size = st.sidebar.selectbox(
    "Batch Size",
    [16,32,64],
    index=1
)

uploaded_file = st.sidebar.file_uploader(
    "/content/Bitcoin Historical Data (1).csv",
    type=["csv"]
)

run = st.sidebar.button("Train Model")

# --------------------------------------------------
# FUNCTIONS
# --------------------------------------------------

def create_sequences(data, window_size=60, horizon=3):

    X=[]
    y=[]

    for i in range(window_size,len(data)-horizon+1):
        X.append(data[i-window_size:i])
        y.append(data[i:i+horizon])

    return np.array(X),np.array(y)


def build_lstm(input_shape,horizon):

    model=models.Sequential([
        layers.LSTM(64,input_shape=input_shape),
        layers.Dense(32,activation="relu"),
        layers.Dense(horizon)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


def build_rnn(input_shape,horizon):

    model=models.Sequential([
        layers.SimpleRNN(50,input_shape=input_shape),
        layers.Dense(horizon)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


def build_cnn(input_shape,horizon):

    model=models.Sequential([

        layers.Conv1D(
            filters=64,
            kernel_size=3,
            activation="relu",
            input_shape=input_shape
        ),

        layers.MaxPooling1D(2),

        layers.Flatten(),

        layers.Dense(50,activation="relu"),

        layers.Dense(horizon)

    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


def build_transformer(input_shape,horizon):

    inputs=layers.Input(shape=input_shape)

    attention=layers.MultiHeadAttention(
        num_heads=4,
        key_dim=input_shape[-1]
    )(inputs,inputs)

    x=layers.LayerNormalization()(attention+inputs)

    x=layers.GlobalAveragePooling1D()(x)

    x=layers.Dense(
        64,
        activation="relu"
    )(x)

    outputs=layers.Dense(horizon)(x)

    model=models.Model(inputs,outputs)

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


# --------------------------------------------------
# TRAIN
# --------------------------------------------------

if uploaded_file is not None:

    df=pd.read_csv(uploaded_file)

    st.subheader("Dataset Preview")

    st.dataframe(df.head())

    df["Price"]=df["Price"].str.replace(",","").astype(float)

    data=df[["Price"]].values

    scaler=MinMaxScaler()

    scaled_data=scaler.fit_transform(data)

    X,y=create_sequences(
        scaled_data,
        window_size=60,
        horizon=3
    )

    split=int(len(X)*0.8)

    X_train=X[:split]
    X_test=X[split:]

    y_train=y[:split]
    y_test=y[split:]

    if run:

        st.info("Training Model...")

        if selected_model=="LSTM":

            model=build_lstm((60,1),3)

        elif selected_model=="RNN":

            model=build_rnn((60,1),3)

        elif selected_model=="CNN":

            model=build_cnn((60,1),3)

        else:

            model=build_transformer((60,1),3)

        history=model.fit(

            X_train,
            y_train,

            validation_data=(X_test,y_test),

            epochs=epochs,

            batch_size=batch_size,

            verbose=0

        )

        pred=model.predict(X_test)

        pred_actual=scaler.inverse_transform(
            pred.reshape(-1,1)
        )

        y_actual=scaler.inverse_transform(
            y_test.reshape(-1,1)
        )

        mae=mean_absolute_error(
            y_actual,
            pred_actual
        )

        rmse=np.sqrt(
            mean_squared_error(
                y_actual,
                pred_actual
            )
        )

        mape=mean_absolute_percentage_error(
            y_actual,
            pred_actual
        )*100

        st.success("Training Completed")

        # -----------------------------
        # METRICS
        # -----------------------------

        st.subheader("Evaluation Metrics")

        c1,c2,c3=st.columns(3)

        c1.metric("MAE",f"{mae:.2f}")

        c2.metric("RMSE",f"{rmse:.2f}")

        c3.metric("MAPE",f"{mape:.2f}%")

        # -----------------------------
        # ACTUAL VS PREDICTED
        # -----------------------------

        st.subheader("Actual vs Predicted")

        fig,ax=plt.subplots(figsize=(12,5))

        ax.plot(
            y_actual,
            label="Actual",
            color="blue"
        )

        ax.plot(
            pred_actual,
            label="Predicted",
            color="red"
        )

        ax.legend()

        st.pyplot(fig)

        # -----------------------------
        # LOSS CURVE
        # -----------------------------

        st.subheader("Training Loss")

        fig2,ax2=plt.subplots(figsize=(12,5))

        ax2.plot(
            history.history["loss"],
            label="Train Loss"
        )

        ax2.plot(
            history.history["val_loss"],
            label="Validation Loss"
        )

        ax2.legend()

        st.pyplot(fig2)

        # -----------------------------
        # FUTURE PREDICTION
        # -----------------------------

        st.subheader("Next 3-Day Prediction")

        last60=scaled_data[-60:]

        future=model.predict(
            last60.reshape(1,60,1)
        )

        future=scaler.inverse_transform(
            future.reshape(-1,1)
        )

        future=pd.DataFrame({

            "Day":[
                "Day 1",
                "Day 2",
                "Day 3"
            ],

            "Predicted Price $":future.flatten()

        })

        st.table(future)
# -----------------------------
# Footer
# -----------------------------

st.sidebar.markdown("---")
st.sidebar.write("Created by Dhanush DB")


In [ ]:
!streamlit run bitcoinapp.py & npx localtunnel --port 8501